In [ ]:
MODEL_CODE = "svm"
MODEL_NAME = "SVM"

# 06: SVM Modeli 

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini · **Model ailesi:** mesafe / sınır tabanlı

**SVM (Destek Vektör Makinesi)** iki sınıfı birbirinden en geniş boşlukla (marjin) ayıran sınırı arar. `rbf` kernel ile doğrusal olmayan, eğri sınırlar da çizebilir. Mesafe hesabına dayandığı için ölçeklemeye duyarlıdır.

Ekip kuralı gereği düz `SVC()` kullanılır (`probability=True` gerekmez, eğitimi ~5 kat yavaşlatır). Ortak fonksiyonlar `predict_proba` olmayan modellerde skor olarak `sigmoid(decision_function)` kullanır: bu bir olasılık değil, 0 ile 1 arasında bir **risk skorudur**.

**Bu notebook'ta yapılanlar:** hiperparametre araması · C'nin overfitting'e etkisi · SMOTE ve class_weight karşılaştırması · feature engineering etkisi · CV değerlendirmesi ve eşik seçimi · öğrenme eğrisi · test seti · XAI

**Süre:** SVM bu notebook'taki en yavaş model; Colab'da tüm notebook yaklaşık 10 ila 15 dakika sürer.

## 1. Kurulum

In [ ]:
# shap and lime are not preinstalled on Colab
try:
    import lime
    import shap
except ImportError:
    %pip install -q shap lime

In [ ]:
import sys
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

# Çıktı klasörleri: repo kökünde models/ ve figures/ (Colab'da çalışma klasöründe)
ROOT = SRC_DIR.resolve().parent if SRC_DIR.name == "src" else Path(".")
MODEL_DIR = ROOT / "models";  MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR = ROOT / "figures";   FIG_DIR.mkdir(exist_ok=True)

import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

from config import MAIN_METRIC, RANDOM_STATE, TARGET_RECALL
from preprocessing import build_pipeline, load_data
from metrics import (SCORING, cv_evaluate, learning_curve_analysis, shared_cv, test_evaluate,
                     threshold_analysis)
from explain import explain

warnings.filterwarnings("ignore")

## 2. Veri

 `train.csv` / `test.csv`. Ön işleme (feature engineering, encoding, ölçekleme, SMOTE) `build_pipeline()` içinde yapılır; test seti en sona kadar kullanılmaz.

In [ ]:
X_train, X_test, y_train, y_test = load_data()
print("Eğitim:", X_train.shape, f"| inme: {y_train.sum()} kişi ({y_train.mean():.2%})")
print("Test:  ", X_test.shape, f"| inme: {y_test.sum()} kişi ({y_test.mean():.2%})")

## 3. Hiperparametre araması

| Parametre | Değerler | Anlamı |
|---|---|---|
| `kernel` | linear, rbf | Sınırın şekli: düz ya da esnek |
| `C` | linear: 0.1, 1 · rbf: 0.1, 1, 10 | Hatalara tolerans. Büyük C eğitim verisine sıkı uyar (overfitting riski), küçük C daha geniş marjinli, genel bir model verir |
| `gamma` (rbf) | scale, 0.01 | Tek bir noktanın etki alanı. Büyük gamma çok kıvrımlı sınır, overfitting |

Ortak katlar (`shared_cv()`) ve ortak metrikler (`SCORING`) kullanılır; en iyi model `MAIN_METRIC` (PR-AUC) ile seçilir.

In [ ]:
def search_table(search, params):
    """Hiperparametre denemeleri: eğitim ve CV skoru, aradaki fark (overfitting göstergesi)."""
    r = pd.DataFrame(search.cv_results_)
    t = r[[f"param_{p}" for p in params]].copy()
    t.columns = [p.replace("model__", "") for p in params]
    for m in [MAIN_METRIC, "recall", "roc_auc"]:
        t[f"{m}_cv"] = r[f"mean_test_{m}"]
    t[f"{MAIN_METRIC}_train"] = r[f"mean_train_{MAIN_METRIC}"]
    t["fark (train - cv)"] = t[f"{MAIN_METRIC}_train"] - t[f"{MAIN_METRIC}_cv"]
    return t.sort_values(f"{MAIN_METRIC}_cv", ascending=False).round(4)

In [ ]:
params = [
    {"model__kernel": ["linear"], "model__C": [0.1, 1]},
    {"model__kernel": ["rbf"], "model__C": [0.1, 1, 10], "model__gamma": ["scale", 0.01]},
]
start = time.time()
search = GridSearchCV(build_pipeline(SVC(random_state=RANDOM_STATE), balancing="smote"), params,
                      cv=shared_cv(), scoring=SCORING, refit=MAIN_METRIC,
                      n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)
best = search.best_estimator_

print(f"Süre: {time.time() - start:.0f} sn")
print("En iyi parametreler:", search.best_params_)
print(f"En iyi CV {MAIN_METRIC}: {search.best_score_:.4f}")
search_table(search, ["model__kernel", "model__C", "model__gamma"])

### C'nin etkisi (overfitting)

C büyüdükçe model eğitim verisine daha sıkı uyar ve eğitim skoru artar. CV skoru aynı oranda artmıyorsa model ezberlemeye başlamıştır. Linear ve rbf kernel'in CV skorları yakınsa, verideki ilişki büyük ölçüde doğrusaldır.

In [ ]:
r = pd.DataFrame(search.cv_results_)
r["param_model__gamma"] = r["param_model__gamma"].astype(str)
fig, ax = plt.subplots(figsize=(7, 4.5))
for g in ["scale", "0.01"]:
    sub = r[(r["param_model__kernel"] == "rbf") & (r["param_model__gamma"] == g)]
    ax.plot(sub["param_model__C"].astype(float), sub[f"mean_train_{MAIN_METRIC}"], "o--", label=f"eğitim (gamma={g})")
    ax.plot(sub["param_model__C"].astype(float), sub[f"mean_test_{MAIN_METRIC}"], "o-", label=f"CV (gamma={g})")
ax.set_xscale("log")
ax.set_xlabel("C (log ölçek)")
ax.set_ylabel(MAIN_METRIC)
ax.set_title("SVM (rbf): C ve gamma'nın etkisi")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / f"{MODEL_CODE}_c_overfitting.png", dpi=200)
plt.show()

## 4. Dengeleme yöntemi: SMOTE mu, class_weight mı?

SVM'de iki seçenek var: **SMOTE** (azınlık sınıfı için sentetik hasta üretir) ve **class_weight** (gerçek inme vakalarındaki hatayı daha ağır cezalandırır, sentetik veri üretmez). En iyi parametrelerle ikisi karşılaştırılır.

In [ ]:
best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
rows = {}
for balancing in ["smote", "class_weight"]:
    res = cv_evaluate(build_pipeline(SVC(random_state=RANDOM_STATE, **best_params), balancing=balancing),
                      X_train, y_train, model_name=f"SVM ({balancing})", save=False, verbose=False)
    rows[balancing] = res["summary"]["cv_mean"]
balance_table = pd.DataFrame(rows).T.round(3)
balance_table

## 5. Feature engineering'in etkisi

Ortak pipeline 5 yeni özellik ekler (17 → 28 sütun). Bunların çoğu tansiyondan türetildiği için birbirine çok benzer. Mesafe tabanlı modellerde gereksiz sütunlar mesafeyi bozabilir. En iyi parametrelerle feature engineering açık ve kapalı karşılaştırılır (bu deneyler `results.csv`'ye yazılmaz).

In [ ]:
rows = {}
for fe in [True, False]:
    res = cv_evaluate(build_pipeline(SVC(random_state=RANDOM_STATE, **best_params), balancing="smote",
                                     feature_engineering=fe),
                      X_train, y_train, model_name=f"SVM (FE={fe})", save=False, verbose=False)
    rows["FE açık" if fe else "FE kapalı"] = res["summary"]["cv_mean"]
pd.DataFrame(rows).T.round(3)

## 6. Final model: CV değerlendirmesi

Seçilen model ortak `cv_evaluate` ile değerlendirilir ve sonucu `results.csv`'ye **bir satır** olarak eklenir. Karşılaştırma notebook'u bu satırı kullanır. Eğitim ve CV skorları arasındaki fark (`gap`) overfitting göstergesidir.

In [ ]:
# class_weight, SMOTE'tan daha iyi PR-AUC verdiyse final model onunla kurulur
if balance_table.loc["class_weight", MAIN_METRIC] > balance_table.loc["smote", MAIN_METRIC]:
    best = build_pipeline(SVC(random_state=RANDOM_STATE, **best_params), balancing="class_weight")
    chosen_balancing = "class_weight"
else:
    chosen_balancing = "smote"
print("Seçilen dengeleme:", chosen_balancing)

cv_result = cv_evaluate(best, X_train, y_train, model_name=MODEL_NAME,
                        note=f"{chosen_balancing} + SVM, GridSearch: {best_params}")

## 7. Eşik seçimi

Varsayılan eşik 0.5'tir, ama inmeyi kaçırmak yanlış alarmdan pahalı olduğu için eşik, **test setine bakılmadan** CV tahminleri (OOF) üzerinde seçilir: hedef recall'u (%80) en az yanlış alarmla sağlayan eşik.

In [ ]:
table, threshold = threshold_analysis(cv_result["y"], cv_result["oof_score"], model_name=MODEL_NAME)

## 8. Overfitting kontrolü: öğrenme eğrisi

Eğitim verisi büyüdükçe eğitim ve CV skorları birbirine yaklaşıyorsa model genelleşiyor demektir. Büyük ve kapanmayan bir fark overfitting, ikisinin birlikte düşük kalması underfitting gösterir.

In [ ]:
lc = learning_curve_analysis(best, X_train, y_train, model_name=MODEL_NAME)
lc.round(3)

## 9. Test seti (yalnızca bir kez)

Final model tüm eğitim verisiyle yeniden eğitilir, seçilen eşikle test setinde değerlendirilir ve `models/` klasörüne kaydedilir.

In [ ]:
best.fit(X_train, y_train)
test_result = test_evaluate(best, X_test, y_test, threshold=threshold,
                            model_name=MODEL_NAME, cv_result=cv_result)

joblib.dump(best, MODEL_DIR / f"{MODEL_CODE}.joblib")
print(f"Model kaydedildi: models/{MODEL_CODE}.joblib")

## 10. XAI: modelin açıklanması

SVM iç yapısı doğrudan okunamayan bir model olduğu için ortak `explain()` fonksiyonu **KernelExplainer** kullanır. Bu yöntem yavaş olduğundan test setinden 100 hasta, eğitim verisinin 20 küme merkezi ve hasta başına 300 özellik kombinasyonu kullanılır (yaklaşık 1 ila 3 dakika). XAI fonksiyonuna aramanın kendisi değil, `search.best_estimator_` (`best`) verilir.

**Grafik okuma:** Bar grafiğinde çubuk uzunluğu önemi gösterir. Özet grafiğinde kırmızı = özelliğin değeri yüksek, mavi = düşük; nokta sağdaysa inme riskini artırıyor. LIME'da yeşil çubuklar "inme var" kararını destekleyen koşullardır.

In [ ]:
# explain() grafikleri sırayla gösterir: SHAP bar, SHAP özet, LIME.
# Her grafik gösterilirken figures/ klasörüne de kaydedilir.
_names = iter([f"{MODEL_CODE}_shap_bar.png", f"{MODEL_CODE}_shap_summary.png", f"{MODEL_CODE}_lime.png"])
_show = plt.show
def _save_and_show(*args, **kwargs):
    name = next(_names, None)
    if name:
        plt.gcf().savefig(FIG_DIR / name, dpi=200, bbox_inches="tight")
    _show(*args, **kwargs)

plt.show = _save_and_show
try:
    start = time.time()
    xai = explain(best, X_train, X_test, model_name=MODEL_NAME,
                  sample_size=100, background_size=20, kernel_nsamples=300)
    print(f"\nXAI süresi: {time.time() - start:.0f} sn")
finally:
    plt.show = _show
print("Grafikler kaydedildi:", [p.name for p in sorted(FIG_DIR.glob(f"{MODEL_CODE}_*.png"))])

In [ ]:
xai["shap_importance"].head(10)